# Homework 6.2 - Step 2

## Frozen Splits, Nested Anchors, and the Allowed-Information Regime

Homework 6.2 must use the same decipherment problem as Homework 5.1.

This notebook therefore **reuses**, rather than regenerates:

- the 70% / 15% / 15% train-validation-test glyph split;
- the nested 5%, 10%, 20%, and 40% anchor sets;
- the Homework 5.1 source-token vocabulary;
- the Homework 5.1 Republic-informed pseudo-label maps;
- the evaluation-only hidden bijection.

The purpose of this notebook is to create one frozen Homework 6.2 manifest so every later notebook uses exactly the same artifacts.

No model is trained here.

## Information restrictions

The main comparison retains the Homework 5.1 information regime.

### Permitted during fitting

- training glyph sequence;
- revealed anchor translations;
- the frozen Republic-derived English information;
- pseudo-label maps derived from the permitted Homework 5.1 inference procedure;
- validation glyphs and validation pseudo-labels for model selection.

### Not permitted during fitting or model selection

- unrevealed *Meditations* plaintext;
- unrevealed entries of the true inverse map

$$
\Phi^{-1};
$$

- held-out test labels.

The evaluation-only hidden map is copied only as an **evaluation artifact** and later notebooks must not load it until all models and hyperparameters are fixed.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil

import numpy as np
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

HW6_DATA_DIR = PROJECT_ROOT / "data" / "hw6_2"
CARRY_DIR = HW6_DATA_DIR / "carried_over_hw5_1"

for path in [
    HW6_DATA_DIR,
    CARRY_DIR / "splits",
    CARRY_DIR / "anchors",
    CARRY_DIR / "pseudo_label_maps",
    CARRY_DIR / "evaluation_only",
]:
    path.mkdir(parents=True, exist_ok=True)

# Typical project arrangements are supported:
# 1) sibling folders HW5.1/ and HW6.2/
# 2) an hw5_1 subfolder under the current data folder
# 3) Homework 5.1 files copied directly into a local data folder
candidate_hw5_data_dirs = [
    PROJECT_ROOT.parent / "HW5.1" / "data",
    PROJECT_ROOT.parent / "HW5_1" / "data",
    PROJECT_ROOT / "data" / "hw5_1",
    PROJECT_ROOT / "data" / "HW5_1",
    PROJECT_ROOT / "hw5_1" / "data",
]

HW5_DATA_DIR = next(
    (
        path
        for path in candidate_hw5_data_dirs
        if path.exists()
    ),
    None,
)

if HW5_DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate the Homework 5.1 data folder automatically. "
        "Set HW5_DATA_DIR manually in this cell to the submitted HW5.1/data folder."
    )

print("Project root:", PROJECT_ROOT.resolve())
print("Homework 5.1 data:", HW5_DATA_DIR.resolve())
print("Homework 6.2 data:", HW6_DATA_DIR.resolve())

Project root: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2
Homework 5.1 data: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data
Homework 6.2 data: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2


## Required Homework 5.1 artifacts

The following filenames are fixed by the proofchecked Homework 5.1 submission:

- `source_syllable_vocabulary.csv`
- `decipherment_inputs/splits/train_glyph_sequence.csv`
- `decipherment_inputs/splits/validation_glyph_sequence.csv`
- `decipherment_inputs/splits/test_glyph_sequence.csv`
- `decipherment_inputs/anchors/anchors_05.csv`
- `decipherment_inputs/anchors/anchors_10.csv`
- `decipherment_inputs/anchors/anchors_20.csv`
- `decipherment_inputs/anchors/anchors_40.csv`
- `evaluation_only/hidden_syllable_glyph_bijection.csv`

The Republic-informed pseudo-label maps may have submission-specific filenames, so they are discovered by schema and anchor level below.

In [2]:
required_paths = {
    "source_vocabulary": (
        HW5_DATA_DIR
        / "source_syllable_vocabulary.csv"
    ),
    "train_split": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "splits"
        / "train_glyph_sequence.csv"
    ),
    "validation_split": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "splits"
        / "validation_glyph_sequence.csv"
    ),
    "test_split": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "splits"
        / "test_glyph_sequence.csv"
    ),
    "anchor_05": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "anchors"
        / "anchors_05.csv"
    ),
    "anchor_10": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "anchors"
        / "anchors_10.csv"
    ),
    "anchor_20": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "anchors"
        / "anchors_20.csv"
    ),
    "anchor_40": (
        HW5_DATA_DIR
        / "decipherment_inputs"
        / "anchors"
        / "anchors_40.csv"
    ),
    "hidden_map": (
        HW5_DATA_DIR
        / "evaluation_only"
        / "hidden_syllable_glyph_bijection.csv"
    ),
}

missing = [
    f"{name}: {path}"
    for name, path in required_paths.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required Homework 5.1 artifact(s):\n"
        + "\n".join(missing)
    )

for name, path in required_paths.items():
    print(f"{name:<20}", path.resolve())

source_vocabulary    C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\source_syllable_vocabulary.csv
train_split          C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\train_glyph_sequence.csv
validation_split     C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\validation_glyph_sequence.csv
test_split           C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\splits\test_glyph_sequence.csv
anchor_05            C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_05.csv
anchor_10            C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\decipherment_inputs\anchors\anchors_10.csv
anchor_20            C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2

## Schema helpers

Homework 5.1 uses integer glyph IDs and source-token IDs. The helpers below identify the relevant columns without changing their values.

In [3]:
def pick_column(frame, candidates, label):
    lowered = {
        column.lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]

    # Conservative substring fallback.
    for column in frame.columns:
        low = column.lower()

        if any(
            candidate.lower() in low
            for candidate in candidates
        ):
            return column

    raise KeyError(
        f"Could not identify {label} column. "
        f"Available columns: {list(frame.columns)}"
    )


GLYPH_COLUMN_CANDIDATES = [
    "glyph_id",
    "glyph",
    "glyph_index",
    "token",
]

SOURCE_COLUMN_CANDIDATES = [
    "source_token_id",
    "syllable_id",
    "source_id",
    "token_id",
    "source_token",
]

POSITION_COLUMN_CANDIDATES = [
    "position",
    "token_position",
    "index",
    "sample_index",
]


def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

# Step 2A: copy the frozen split and anchor artifacts

The copies below are byte-for-byte carried-over inputs for Homework 6.2.

In [4]:
copy_targets = {
    "source_vocabulary": (
        CARRY_DIR
        / "source_syllable_vocabulary.csv"
    ),
    "train_split": (
        CARRY_DIR
        / "splits"
        / "train_glyph_sequence.csv"
    ),
    "validation_split": (
        CARRY_DIR
        / "splits"
        / "validation_glyph_sequence.csv"
    ),
    "test_split": (
        CARRY_DIR
        / "splits"
        / "test_glyph_sequence.csv"
    ),
    "anchor_05": (
        CARRY_DIR
        / "anchors"
        / "anchors_05.csv"
    ),
    "anchor_10": (
        CARRY_DIR
        / "anchors"
        / "anchors_10.csv"
    ),
    "anchor_20": (
        CARRY_DIR
        / "anchors"
        / "anchors_20.csv"
    ),
    "anchor_40": (
        CARRY_DIR
        / "anchors"
        / "anchors_40.csv"
    ),
    "hidden_map": (
        CARRY_DIR
        / "evaluation_only"
        / "hidden_syllable_glyph_bijection.csv"
    ),
}

for name, source in required_paths.items():
    target = copy_targets[name]
    shutil.copy2(source, target)

    if sha256(source) != sha256(target):
        raise RuntimeError(
            f"Hash mismatch while copying {name}."
        )

print("All fixed Homework 5.1 artifacts copied with matching SHA-256 hashes.")

All fixed Homework 5.1 artifacts copied with matching SHA-256 hashes.


# Step 2B: verify the same train, validation, and test split

The source glyph sequence has

$$
77{,}260
$$

syllable-token occurrences in Homework 5.1.

The split remains contiguous and unchanged.

In [5]:
split_frames = {}

for split_name in [
    "train",
    "validation",
    "test",
]:
    path = (
        CARRY_DIR
        / "splits"
        / f"{split_name}_glyph_sequence.csv"
    )

    frame = pd.read_csv(path)

    glyph_col = pick_column(
        frame,
        GLYPH_COLUMN_CANDIDATES,
        "glyph ID",
    )

    split_frames[split_name] = {
        "frame": frame,
        "glyph_col": glyph_col,
    }

    print(
        f"{split_name:<11}",
        f"rows={len(frame):,}",
        f"distinct_glyphs={frame[glyph_col].nunique():,}",
    )


total_rows = sum(
    len(item["frame"])
    for item in split_frames.values()
)

split_summary = pd.DataFrame([
    {
        "split": split_name,
        "rows": len(item["frame"]),
        "fraction_of_total": (
            len(item["frame"])
            / total_rows
        ),
        "distinct_glyphs": (
            item["frame"][
                item["glyph_col"]
            ].nunique()
        ),
    }
    for split_name, item
    in split_frames.items()
])

split_summary_path = (
    HW6_DATA_DIR
    / "task2_split_manifest.csv"
)

split_summary.to_csv(
    split_summary_path,
    index=False,
)

print()
print(split_summary.to_string(index=False))
print()
print("Total rows:", f"{total_rows:,}")
print("Saved:", split_summary_path.resolve())

train       rows=54,082 distinct_glyphs=2,671
validation  rows=11,589 distinct_glyphs=1,305
test        rows=11,589 distinct_glyphs=1,365

     split  rows  fraction_of_total  distinct_glyphs
     train 54082               0.70             2671
validation 11589               0.15             1305
      test 11589               0.15             1365

Total rows: 77,260
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2\task2_split_manifest.csv


# Step 2C: verify the nested anchor sets

Homework 5.1 used one seeded ordering of glyph types. The 5%, 10%, 20%, and 40% anchor maps are nested:

$$
A_5
\subseteq
A_{10}
\subseteq
A_{20}
\subseteq
A_{40}.
$$

No anchors are resampled in Homework 6.2.

In [6]:
anchor_levels = ["05", "10", "20", "40"]

anchor_frames = {}
anchor_sets = {}
anchor_summary_rows = []

for level in anchor_levels:
    path = (
        CARRY_DIR
        / "anchors"
        / f"anchors_{level}.csv"
    )

    frame = pd.read_csv(path)

    glyph_col = pick_column(
        frame,
        GLYPH_COLUMN_CANDIDATES,
        "glyph ID",
    )

    source_col = pick_column(
        frame,
        SOURCE_COLUMN_CANDIDATES,
        "source-token ID",
    )

    if frame[glyph_col].duplicated().any():
        raise ValueError(
            f"Duplicate glyph IDs in anchors_{level}.csv"
        )

    anchor_frames[level] = {
        "frame": frame,
        "glyph_col": glyph_col,
        "source_col": source_col,
    }

    anchor_sets[level] = set(
        frame[glyph_col].tolist()
    )

    anchor_summary_rows.append({
        "anchor_level": int(level),
        "revealed_glyph_types": len(frame),
    })


for small, large in [
    ("05", "10"),
    ("10", "20"),
    ("20", "40"),
]:
    if not (
        anchor_sets[small]
        <= anchor_sets[large]
    ):
        raise ValueError(
            f"Anchor nesting failed: "
            f"A_{small} is not a subset of A_{large}."
        )


source_vocab = pd.read_csv(
    CARRY_DIR
    / "source_syllable_vocabulary.csv"
)

K = len(source_vocab)

anchor_summary = pd.DataFrame(
    anchor_summary_rows
)

anchor_summary[
    "actual_percentage"
] = (
    100.0
    * anchor_summary[
        "revealed_glyph_types"
    ]
    / K
)

anchor_summary_path = (
    HW6_DATA_DIR
    / "task2_anchor_summary.csv"
)

anchor_summary.to_csv(
    anchor_summary_path,
    index=False,
)

print(anchor_summary.to_string(index=False))
print()
print("Nested anchor condition verified.")
print("Source vocabulary size K =", K)

 anchor_level  revealed_glyph_types  actual_percentage
            5                   153           5.013106
           10                   305           9.993447
           20                   610          19.986894
           40                  1221          40.006553

Nested anchor condition verified.
Source vocabulary size K = 3052


# Step 2D: discover and freeze the Homework 5.1 Republic-informed pseudo-label maps

Homework 5.1 generated a static candidate inverse map for each anchor level using:

- training glyph bigram counts;
- the fixed smoothed *Republic* bigram model;
- anchor constraints;
- the same seeded inference procedure.

Homework 6.2 must not secretly create stronger labels for the attention decoder. Therefore both the attention and non-attention decoders will use the same carried-over pseudo-label map at each anchor level.

The search below looks for a full glyph-to-source map for each anchor percentage while excluding the anchor files and the evaluation-only hidden answer key.

In [7]:
def candidate_csvs_for_level(level):
    level_tokens = {
        level,
        str(int(level)),
        f"{int(level):02d}",
        f"{int(level)}pct",
        f"{int(level)}percent",
    }

    candidates = []

    for path in HW5_DATA_DIR.rglob("*.csv"):
        low_name = path.name.lower()
        low_full = str(path).lower()

        if "anchor" in low_full:
            continue

        if "evaluation_only" in low_full:
            continue

        if "hidden_syllable_glyph_bijection" in low_name:
            continue

        if not any(
            token in low_name
            for token in level_tokens
        ):
            continue

        if not any(
            keyword in low_name
            for keyword in [
                "infer",
                "pseudo",
                "candidate",
                "map",
            ]
        ):
            continue

        try:
            frame = pd.read_csv(path)

            glyph_col = pick_column(
                frame,
                GLYPH_COLUMN_CANDIDATES,
                "glyph ID",
            )

            source_col = pick_column(
                frame,
                SOURCE_COLUMN_CANDIDATES,
                "source-token ID",
            )

        except Exception:
            continue

        # A full candidate map should cover essentially the whole glyph vocabulary.
        if frame[glyph_col].nunique() < int(0.95 * K):
            continue

        candidates.append({
            "path": path,
            "frame": frame,
            "glyph_col": glyph_col,
            "source_col": source_col,
        })

    return candidates


pseudo_map_manifest = {}

for level in anchor_levels:
    candidates = candidate_csvs_for_level(
        level
    )

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not automatically locate the full Homework 5.1 "
            f"Republic-informed pseudo-label map for {int(level)}% anchors.\n"
            "Search your HW5.1/data folder for the candidate/inferred map CSV "
            "and add its exact path manually in this cell."
        )

    # Prefer the shortest path/name as the most direct artifact if duplicates exist.
    candidates = sorted(
        candidates,
        key=lambda item: (
            len(str(item["path"])),
            str(item["path"]),
        ),
    )

    selected = candidates[0]

    source_path = selected["path"]
    target_path = (
        CARRY_DIR
        / "pseudo_label_maps"
        / f"pseudo_map_{level}.csv"
    )

    normalized = selected["frame"][
        [
            selected["glyph_col"],
            selected["source_col"],
        ]
    ].copy()

    normalized.columns = [
        "glyph_id",
        "source_token_id",
    ]

    normalized = (
        normalized
        .drop_duplicates(
            subset=["glyph_id"]
        )
        .sort_values("glyph_id")
        .reset_index(drop=True)
    )

    if normalized["glyph_id"].nunique() != K:
        raise ValueError(
            f"Pseudo-label map for {level}% anchors "
            f"has {normalized['glyph_id'].nunique()} glyphs, expected {K}."
        )

    normalized.to_csv(
        target_path,
        index=False,
    )

    pseudo_map_manifest[level] = {
        "source_path": str(
            source_path.resolve()
        ),
        "copied_path": str(
            target_path.relative_to(
                PROJECT_ROOT
            )
        ),
        "rows": len(normalized),
    }

    print(
        f"{level}% pseudo-label map:",
        source_path.resolve(),
    )

05% pseudo-label map: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\inferred_maps\inferred_map_05.csv
10% pseudo-label map: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\inferred_maps\inferred_map_10.csv
20% pseudo-label map: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\inferred_maps\inferred_map_20.csv
40% pseudo-label map: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW5.1\data\inferred_maps\inferred_map_40.csv


## Anchor consistency check for the pseudo-label maps

At every anchor level, a revealed glyph must carry exactly its revealed source-token label in the pseudo-label map.

In [8]:
for level in anchor_levels:
    anchor = anchor_frames[level]
    pseudo = pd.read_csv(
        CARRY_DIR
        / "pseudo_label_maps"
        / f"pseudo_map_{level}.csv"
    )

    anchor_pairs = (
        anchor["frame"][
            [
                anchor["glyph_col"],
                anchor["source_col"],
            ]
        ]
        .rename(
            columns={
                anchor["glyph_col"]: "glyph_id",
                anchor["source_col"]: "source_token_id",
            }
        )
    )

    merged = anchor_pairs.merge(
        pseudo,
        on="glyph_id",
        suffixes=(
            "_anchor",
            "_pseudo",
        ),
        how="left",
        validate="one_to_one",
    )

    mismatch = (
        merged[
            "source_token_id_anchor"
        ]
        != merged[
            "source_token_id_pseudo"
        ]
    )

    if mismatch.any():
        raise ValueError(
            f"Pseudo-label map violates revealed anchors at {level}%."
        )

    print(
        f"{level}% anchor consistency:",
        "verified",
    )

05% anchor consistency: verified
10% anchor consistency: verified
20% anchor consistency: verified
40% anchor consistency: verified


# Step 2E: freeze common validation/test center positions

Later architecture experiments use context radii up to

$$
m_{\max}=16,
$$

corresponding to a 33-glyph centered window.

To make model comparisons like-for-like, the common evaluation positions are fixed now:

$$
m_{\max}
\le i
<
N-m_{\max}.
$$

This prevents a wider-context model from being evaluated on a different set of positions than a narrower-context model.

In [9]:
MAX_CONTEXT_RADIUS = 16


def common_positions(length, radius):
    return np.arange(
        radius,
        length - radius,
        dtype=int,
    )


position_rows = []

for split_name in [
    "validation",
    "test",
]:
    n = len(
        split_frames[split_name]["frame"]
    )

    positions = common_positions(
        n,
        MAX_CONTEXT_RADIUS,
    )

    for position in positions:
        position_rows.append({
            "split": split_name,
            "position": int(position),
        })


common_positions_frame = pd.DataFrame(
    position_rows
)

common_positions_path = (
    HW6_DATA_DIR
    / "task2_common_evaluation_positions.csv"
)

common_positions_frame.to_csv(
    common_positions_path,
    index=False,
)

print(
    common_positions_frame
    .groupby("split")
    .size()
)
print()
print("Saved:", common_positions_path.resolve())

split
test          11557
validation    11557
dtype: int64

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2\task2_common_evaluation_positions.csv


# Step 2F: save the frozen information manifest

The manifest is the single source of truth for all later HW6.2 notebooks.

The evaluation-only hidden map is listed but must not be loaded before final evaluation.

In [10]:
manifest = {
    "homework": "6.2",
    "reuse_hw5_1": True,
    "source_vocabulary_size": K,
    "max_architecture_context_radius": (
        MAX_CONTEXT_RADIUS
    ),
    "splits": {
        split_name: str(
            (
                CARRY_DIR
                / "splits"
                / f"{split_name}_glyph_sequence.csv"
            ).relative_to(
                PROJECT_ROOT
            )
        )
        for split_name
        in [
            "train",
            "validation",
            "test",
        ]
    },
    "anchors": {
        level: str(
            (
                CARRY_DIR
                / "anchors"
                / f"anchors_{level}.csv"
            ).relative_to(
                PROJECT_ROOT
            )
        )
        for level in anchor_levels
    },
    "pseudo_label_maps": {
        level: str(
            (
                CARRY_DIR
                / "pseudo_label_maps"
                / f"pseudo_map_{level}.csv"
            ).relative_to(
                PROJECT_ROOT
            )
        )
        for level in anchor_levels
    },
    "source_vocabulary": str(
        (
            CARRY_DIR
            / "source_syllable_vocabulary.csv"
        ).relative_to(
            PROJECT_ROOT
        )
    ),
    "evaluation_only_hidden_map": str(
        (
            CARRY_DIR
            / "evaluation_only"
            / "hidden_syllable_glyph_bijection.csv"
        ).relative_to(
            PROJECT_ROOT
        )
    ),
    "common_evaluation_positions": str(
        common_positions_path.relative_to(
            PROJECT_ROOT
        )
    ),
    "allowed_information": {
        "training_glyphs": True,
        "revealed_anchors": True,
        "frozen_republic_information": True,
        "carried_over_pseudo_labels": True,
        "unrevealed_meditations_plaintext_for_fitting": False,
        "unrevealed_true_map_for_fitting": False,
        "test_labels_for_fitting_or_selection": False,
    },
    "pseudo_map_source_discovery": (
        pseudo_map_manifest
    ),
}

manifest_path = (
    HW6_DATA_DIR
    / "task2_frozen_information_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", manifest_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW6.2\data\hw6_2\task2_frozen_information_manifest.json


# Step 2 Conclusion

Homework 6.2 now has a frozen experimental regime:

$$
\boxed{
\text{same split}
+
\text{same anchors}
+
\text{same pseudo-label construction}
+
\text{same hidden answer key}
}
$$

The only intended change in the main experiment is the decoder architecture.

Later notebooks must read `task2_frozen_information_manifest.json` rather than rediscovering or resampling these artifacts.